# hydroml — notebook driver

This notebook is a thin front end for the same pipeline that `run_pipeline.py`
executes. All settings live in a YAML config; no analysis code belongs in here.

1. edit `../config.yaml` (or copy it per basin)
2. run the cells below
3. inspect `results/<run_name>/`

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import os
os.chdir(ROOT)

from hydroml import REGISTRY, build_dataset, load_config, run
print("models available:", [k for k, s in REGISTRY.items() if s.available()])

## 1. Inspect the configuration and the prepared data

In [ ]:
CONFIG = "config_demo.yaml"   # switch to "config.yaml" for your basin

cfg = load_config(CONFIG)
ds = build_dataset(cfg)
ds.summary()

In [ ]:
# first rows of the engineered feature matrix
ds.X.head()

## 2. Run the full comparison

Keyword arguments override the YAML, which keeps one config file usable for
quick experiments.

In [ ]:
result = run(CONFIG)
result.leaderboard

In [ ]:
# example override: two models, no tuning, different run name
# result = run(CONFIG,
#              run={"name": "quick_check"},
#              models={"enabled": ["xgboost", "lightgbm"]},
#              tuning={"enabled": False})

## 3. Results

In [ ]:
result.metrics_wide[result.metrics_wide["split"] == "test"]

In [ ]:
result.predictions.head()

## 3b. Diagnostics and cross-validation

`diagnostics` holds the lag/seasonality tables computed on the training rows;
`cv_summary` is populated when `evaluation.cv.enabled` is true. Compare the
single-split leaderboard with the CV means: if the fold-to-fold spread exceeds
the gap between two models, they are indistinguishable on this record.

In [ ]:
result.diagnostics.get("cross_correlation", "not computed")

In [ ]:
result.cv_summary

In [ ]:
from IPython.display import Image, display

for name in ("hydrograph", "scatter", "metric_bars", "residuals",
             "lag_correlation", "seasonality", "importance", "cv_metrics"):
    path = result.run_dir / "figures" / f"{name}.png"
    if path.exists():
        display(Image(filename=str(path)))

## 4. Re-use a fitted model

`result.estimators` holds the fitted objects; `results/<run>/models/` holds the
serialised copies. Predictions must be made on the **scaled** feature matrix and
pushed back through the target scaler — `hydroml.fitting.predict_raw` does both.

In [ ]:
from hydroml.fitting import predict_raw

best = result.leaderboard.loc[0, "model"]
sim = predict_raw(result.estimators[best], result.config, result.dataset,
                  result.dataset.splits["test"])
print(best, sim[:5])